In [0]:
import dlt
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

# ---- Paths for manifests (from your ingest notebook) ----
MANIFESTS_PATH = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/fred_raw/manifests/"

# ---- Schema to parse observations if it's stored as STRING ----
obs_schema = ArrayType(StructType([
    StructField("realtime_start", StringType(), True),
    StructField("realtime_end",   StringType(), True),
    StructField("date",           StringType(), True),
    StructField("value",          StringType(), True),
]))

def _base():
    b = dlt.read_stream("canada_business.bronze.bronze_fred_raw_json")

    # If observations is already an array, keep it. If it's a string, parse it.
    observations_arr = F.when(
        F.col("observations").cast("string").startswith("["),
        F.from_json(F.col("observations").cast("string"), obs_schema)
    ).otherwise(F.lit(None).cast(obs_schema))

    return (
        b.withColumn("observations_arr", observations_arr)
         .withColumn("obs", F.explode_outer(F.col("observations_arr")))
         .withColumn("observation_date", F.to_date(F.col("obs.date")))
         .withColumn("value_raw", F.col("obs.value"))
         .withColumn(
             "value_double",
             F.when(F.col("value_raw").isin(".", "NA", ""), F.lit(None).cast("double"))
              .otherwise(F.col("value_raw").cast("double"))
         )
         .withColumn("obs_realtime_start", F.to_date(F.col("obs.realtime_start")))
         .withColumn("obs_realtime_end",   F.to_date(F.col("obs.realtime_end")))
         .withColumn("realtime_start",     F.to_date(F.col("realtime_start")))
         .withColumn("realtime_end",       F.to_date(F.col("realtime_end")))
    )

# ---------------- Silver: quarantine ----------------
@dlt.table(
  name="silver_fred_quarantine",
  comment="Rows that failed parsing or have missing/invalid fields"
)
def silver_fred_quarantine():
    x = _base()
    return (
        x.select(
            "series_id","_ingest_ts","_source_file",
            "observation_date","value_raw","value_double",
            "obs_realtime_start","obs_realtime_end","realtime_start","realtime_end",
            F.when(F.col("observations_arr").isNull(), F.lit("observations_parse_failed"))
             .when(F.col("obs").isNull(), F.lit("missing_observation_object"))
             .when(F.col("observation_date").isNull(), F.lit("invalid_or_missing_date"))
             .otherwise(F.lit("other")).alias("reject_reason")
        )
        .where(
            F.col("observations_arr").isNull() |
            F.col("obs").isNull() |
            F.col("observation_date").isNull()
        )
    )

# ---------------- Silver: clean observations ----------------
@dlt.table(
  name="silver_fred_observations",
  comment="Clean FRED observations (one row per series_id per date per ingest file)"
)
@dlt.expect_or_drop("valid_series", "series_id IS NOT NULL AND series_id <> ''")
@dlt.expect_or_drop("valid_date",   "observation_date IS NOT NULL")
def silver_fred_observations():
    x = _base()
    return x.select(
        "series_id","observation_date",
        "value_raw","value_double",
        "obs_realtime_start","obs_realtime_end",
        "realtime_start","realtime_end",
        "_ingest_ts","_source_file"
    )

# ---------------- Silver: basic metrics ----------------
@dlt.table(
  name="silver_fred_metrics",
  comment="Basic monitoring metrics"
)
def silver_fred_metrics():
    x = _base()
    return (
        x.groupBy()
         .agg(
             F.count("*").alias("rows_seen"),
             F.sum(F.when(F.col("observations_arr").isNull(), 1).otherwise(0)).alias("observations_parse_failed"),
             F.sum(F.when(F.col("observation_date").isNull(), 1).otherwise(0)).alias("invalid_date"),
             F.max(F.col("_ingest_ts")).alias("latest_ingest_ts")
         )
    )

# ---------------- Silver: ingest run audit (from manifests) ----------------
@dlt.table(
  name="silver_fred_runs",
  comment="One row per ingest run (from run_manifest_*.json)"
)
def silver_fred_runs():
    df = (
        spark.read
          .format("json")
          .option("multiLine", "true")
          .load(MANIFESTS_PATH)
    )

    per_series = df.select(
        "run_ts","catalog","schema","bronze_base","observation_start",
        F.explode_outer("series").alias("s")
    ).select(
        "run_ts","catalog","schema","bronze_base","observation_start",
        F.col("s.series_id").alias("series_id"),
        F.col("s.rows").cast("long").alias("rows"),
        F.col("s.raw_file").alias("raw_file")
    )

    return (
        per_series.groupBy("run_ts","catalog","schema","bronze_base","observation_start")
          .agg(
              F.countDistinct("series_id").alias("series_count"),
              F.countDistinct("raw_file").alias("file_count"),
              F.sum("rows").alias("total_rows")
          )
    )

In [0]:
# ---------------- Silver: Google Trends (pivot + score) ----------------
@dlt.table(
  name="silver_google_trends",
  comment="Pivoted Google Trends (one row per date) + demand_trend_score"
)
def silver_google_trends():
    b = dlt.read("canada_business.bronze.bronze_google_trends_raw")

    p = (
        b.groupBy("date")
         .agg(
             F.max(F.when(F.col("keyword") == "buy laptop", F.col("value"))).alias("laptop_idx"),
             F.max(F.when(F.col("keyword") == "TV deals", F.col("value"))).alias("tv_idx"),
             F.max(F.when(F.col("keyword") == "gaming console", F.col("value"))).alias("console_idx"),
             F.max(F.when(F.col("keyword") == "electronics sale", F.col("value"))).alias("electronics_sale_idx"),
         )
    )

    return (
        p.withColumn(
            "demand_trend_score",
            0.35*F.col("laptop_idx") +
            0.25*F.col("tv_idx") +
            0.25*F.col("console_idx") +
            0.15*F.col("electronics_sale_idx")
        )
        .withColumn("ingest_ts", F.current_timestamp())
    )

In [0]:

from pyspark.sql import functions as F

# ---------------- Silver: WITS tariff (SDMX JSON) ----------------

def _wits_base():
    b = dlt.read_stream("canada_business.bronze.bronze_wits_tariff_raw")

    # Build arrays of dimension IDs (so we can map series_key indices -> real IDs)
    freq_vals = F.expr("transform(j.structure.dimensions.series[0].values, x -> x.id)")
    reporter_vals = F.expr("transform(j.structure.dimensions.series[1].values, x -> x.id)")
    partner_vals = F.expr("transform(j.structure.dimensions.series[2].values, x -> x.id)")
    product_vals = F.expr("transform(j.structure.dimensions.series[3].values, x -> x.id)")
    indicator_vals = F.expr("transform(j.structure.dimensions.series[4].values, x -> x.id)")

    time_vals = F.expr("transform(j.structure.dimensions.observation[0].values, x -> x.id)")  # years

    # series_key looks like "0:0:0:12:0" -> each part indexes one dimension in order above
    key_parts = F.split(F.col("series_key"), ":")

    freq_idx      = F.element_at(key_parts, 1).cast("int")
    reporter_idx  = F.element_at(key_parts, 2).cast("int")
    partner_idx   = F.element_at(key_parts, 3).cast("int")
    product_idx   = F.element_at(key_parts, 4).cast("int")
    indicator_idx = F.element_at(key_parts, 5).cast("int")

    # Map indices -> IDs (element_at is 1-based, so +1)
    freq_id      = F.element_at(freq_vals,      freq_idx + 1)
    reporter_id  = F.element_at(reporter_vals,  reporter_idx + 1)
    partner_id   = F.element_at(partner_vals,   partner_idx + 1)
    product_id   = F.element_at(product_vals,   product_idx + 1)
    indicator_id = F.element_at(indicator_vals, indicator_idx + 1)

    # Explode observations map: {"0":[12.3], "1":[11.8], ...}
    obs_entries = F.map_entries(F.col("series_struct.observations"))
    x = (
        b.withColumn("obs_entry", F.explode_outer(obs_entries))
         .withColumn("time_idx", F.col("obs_entry.key").cast("int"))
         .withColumn("obs_arr",  F.col("obs_entry.value"))
         .withColumn("tariff_value_raw", F.element_at(F.col("obs_arr"), 1))
         .withColumn("year_str", F.element_at(time_vals, F.col("time_idx") + 1))
         .withColumn("year", F.col("year_str").cast("int"))
         .withColumn("freq", freq_id)
         .withColumn("reporter", reporter_id)
         .withColumn("partner", partner_id)
         .withColumn("productcode", product_id)
         .withColumn("indicator", indicator_id)
         # UC-safe file path (works in UC)
         .withColumn("source_file", F.coalesce(F.col("source_file"), F.col("_metadata.file_path")))
         .withColumn("ingest_ts", F.current_timestamp())
    )

    # Cast + basic cleaning
    x = (
        x.withColumn("tariff_value", F.col("tariff_value_raw").cast("double"))
         # Outlier flags (example thresholds — adjust if you want)
         .withColumn("is_negative", F.col("tariff_value").isNotNull() & (F.col("tariff_value") < 0))
         .withColumn("is_extreme",  F.col("tariff_value").isNotNull() & (F.col("tariff_value") > 200))
    )

    return x


@dlt.table(
  name="silver_wits_tariff_quarantine",
  comment="Quarantined WITS tariff rows (missing values, bad year, outliers, parsing gaps)"
)
def silver_wits_tariff_quarantine():
    x = _wits_base()

    return (
        x.withColumn(
            "reject_reason",
            F.when(F.col("year").isNull(), F.lit("missing_or_invalid_year"))
             .when(F.col("tariff_value").isNull(), F.lit("missing_or_invalid_tariff_value"))
             .when(F.col("reporter").isNull() | F.col("partner").isNull() | F.col("productcode").isNull() | F.col("indicator").isNull(),
                   F.lit("missing_dimension_mapping"))
             .when(F.col("is_negative"), F.lit("negative_value_outlier"))
             .when(F.col("is_extreme"),  F.lit("extreme_value_outlier"))
             .otherwise(F.lit(None))
        )
        .where(F.col("reject_reason").isNotNull())
        .select(
            "freq","reporter","partner","productcode","indicator",
            "year","tariff_value_raw","tariff_value",
            "reject_reason","series_key","source_file","ingest_ts"
        )
    )


@dlt.table(
  name="silver_wits_tariff",
  comment="Clean WITS tariff table (one row per reporter/partner/product/indicator/year)"
)
@dlt.expect_or_drop("valid_year", "year IS NOT NULL")
@dlt.expect_or_drop("valid_value", "tariff_value IS NOT NULL")
def silver_wits_tariff():
    x = _wits_base()

    clean = (
        x.where(
            (F.col("year").isNotNull()) &
            (F.col("tariff_value").isNotNull()) &
            (~F.col("is_negative")) &
            (~F.col("is_extreme"))
        )
        # Deduplicate if the same datapoint shows up multiple times
        .dropDuplicates(["reporter","partner","productcode","indicator","year"])
    )

    return clean.select(
        "freq","reporter","partner","productcode","indicator",
        "year","tariff_value",
        "series_key","source_file","ingest_ts"
    )


@dlt.table(
  name="silver_wits_tariff_metrics",
  comment="Data quality metrics for WITS tariff dataset"
)
def silver_wits_tariff_metrics():
    x = _wits_base()

    return (
        x.groupBy()
         .agg(
            F.count("*").alias("rows_seen"),
            F.sum(F.when(F.col("year").isNull(), 1).otherwise(0)).alias("missing_year"),
            F.sum(F.when(F.col("tariff_value").isNull(), 1).otherwise(0)).alias("missing_value"),
            F.sum(F.when(F.col("is_negative"), 1).otherwise(0)).alias("negative_outliers"),
            F.sum(F.when(F.col("is_extreme"), 1).otherwise(0)).alias("extreme_outliers"),
            F.countDistinct("series_key").alias("series_keys"),
            F.max("ingest_ts").alias("latest_ingest_ts")
         )
    )